# OB-GNN Single Custom Sample Prediction
This notebook is for preprocessing and running OB-GNN inference on a single airfoil sample. 

## 1. Imports & Inputs

In [ ]:
import torch
import numpy as np
import os
import airfrans as af
import dgl
from sklearn.neighbors import NearestNeighbors
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from save_predictions import load_model, CHECKPOINT_DIR

# change this to desired input filename for your airfoil sample
SAMPLE_NAME = 'nominal_naca0012'
# hard coded airfoil parameters
AIRFOIL_PARAMS = [4.58, 2.02, 19.22]

base_dir = Path.cwd().parent
# input
input_npy = base_dir / 'custom_samples' / f'{SAMPLE_NAME}.npy'
# preprocessed input
processed_path = base_dir / 'processed_samples' / f'processed_{SAMPLE_NAME}.pt'
# ob-gnn prediction output
out_path = str(base_dir) + f"/predictions/deviated/pred_{SAMPLE_NAME}.pt"
# prediction plot visualizations
out_dir = base_dir / "prediction_plots/nominal"

# hard-coded normalization params from original authors
NORMALIZE_PARAMS = {
    "means": [62, 4.5, 12.5, 62, 5, 42, 10, -475, 0.0008],
    "stds":  [20, 5.5, 4.2, 20, 6.5, 30, 31, 2800, 0.003],
    "maxs":  [0.025, 0.065],
}

/home/jlu25/.conda/envs/ob_gnn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Preprocessing Custom Sample

In [ ]:
def build_naca_shape_feature(airfoil_params: list):
    shape_feature = []

    # ---- geometry params ----
    airfoil_params = np.array(airfoil_params, dtype=np.float32)

    # ---- normalized first scalar ----
    thickness_param = airfoil_params[-1]
    normalized_scalar = (
        (thickness_param - NORMALIZE_PARAMS["means"][0])
        / NORMALIZE_PARAMS["stds"][0]
    )
    shape_feature.append(float(normalized_scalar))

    # ---- camber + thickness ----
    # sample airfoil at 20 evenly spaced locations along the chord
    xlen = 20 
    xs = np.arange(0, 1, 1 / xlen)

    ys, _ = af.naca_generator.camber_line(airfoil_params[:-1], xs)
    rs = af.naca_generator.thickness_dist(airfoil_params[-1] / 100, xs)

    ys = ys / NORMALIZE_PARAMS["maxs"][0]
    rs = rs / NORMALIZE_PARAMS["maxs"][1]

    shape_feature.extend(ys)
    shape_feature.extend(rs)
    shape_feature = torch.tensor(shape_feature, dtype=torch.float32)
    assert shape_feature.shape == (41,)
    return shape_feature


def make_surface_cycle_edges(surface_mask: np.ndarray, x: np.ndarray, y: np.ndarray):
    """
    Sort airfoil-surface nodes by angle around an approximate center,
    forcing trailing edge to index 0, then connect them in a closed cycle.
    
    Returns the cycle edges AND the sorted node indices.
    """
    surface_ids = np.flatnonzero(surface_mask)
    if len(surface_ids) < 2:
        return np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int64)

    surface_pos = np.stack([x[surface_ids], y[surface_ids]], axis=1)

    # estimate center from surface points with x in (0.2, 0.3)
    middle = surface_pos[(surface_pos[:, 0] > 0.2) & (surface_pos[:, 0] < 0.3)]
    if len(middle) > 0:
        cy = 0.5 * (middle[:, 1].max() + middle[:, 1].min())
        center = np.array([0.25, cy], dtype=np.float32)
    else:
        print("no surface points found in 0.2<x<0.3")
        center = surface_pos.mean(axis=0).astype(np.float32)

    offsets = surface_pos - center
    angles = np.arctan2(offsets[:, 1], offsets[:, 0])
    
    # Find the tail, get its angle, and shift all angles so the tail is at 0
    tail_local_idx = np.argmax(surface_pos[:, 0])
    shifted_angles = (angles - angles[tail_local_idx]) % (2 * np.pi)
    order = np.argsort(shifted_angles)

    ordered_ids = surface_ids[order]
    src = ordered_ids
    dst = np.roll(ordered_ids, -1)

    return src.astype(np.int64), dst.astype(np.int64), ordered_ids


def preprocess_custom_sample(
    input_npy_path: str,
    output_pt_path: str,
):
    """
    input_npy_path should contain a numpy object array of length 9:
      0: x-position            (N,)
      1: y-position            (N,)
      2: x-inlet_velocity      (N,)
      3: y-inlet_velocity      (N,)
      4: distance_function     (N,)
      5: x-normals            (N,)
      6: y-normals            (N,)
      7: surface mask         (N,) boolean mask
      8: undirected edges     (E,2)

    Saves:
      output_pt_path -> (graph, shape_features, means, stds)
    """
    arr = np.load(input_npy_path, allow_pickle=True)
    if len(arr) != 9:
        raise ValueError(f"Expected object array of length 9, got {len(arr)}")

    x = np.asarray(arr[0], dtype=np.float32)
    y = np.asarray(arr[1], dtype=np.float32)
    x_in = np.asarray(arr[2], dtype=np.float32)
    y_in = np.asarray(arr[3], dtype=np.float32)
    dist_fn = np.asarray(arr[4], dtype=np.float32)
    x_norm = np.asarray(arr[5], dtype=np.float32)
    y_norm = np.asarray(arr[6], dtype=np.float32)
    surface = np.asarray(arr[7]).astype(bool)
    edges = np.asarray(arr[8], dtype=np.int64)

    n = len(x) # num nodes
    if any(len(v) != n for v in [y, x_in, y_in, dist_fn, x_norm, y_norm, surface]):
        raise ValueError("All node-wise arrays must have the same length N")
    if edges.ndim != 2 or edges.shape[1] != 2:
        raise ValueError(f"edges must have shape (E,2), got {edges.shape}")

    pos = np.stack([x, y], axis=1)

    # ------------------------------------------------------------------
    # Build graph edges:
    #   1) provided mesh edges (undirected -> both directions)
    #   2) airfoil surface cycle
    #   3) kNN edges
    # ------------------------------------------------------------------
    mesh_src = edges[:, 0]
    mesh_dst = edges[:, 1]

    surf_src, surf_dst, ordered_surface_ids = make_surface_cycle_edges(surface, x, y)

    srcs = np.concatenate([mesh_src, mesh_dst,        # both directions
                            surf_src, surf_dst])
    dsts = np.concatenate([mesh_dst, mesh_src,
                                surf_dst, surf_src])
    pos_tensor = torch.from_numpy(pos)
    knn_graph = dgl.knn_graph(pos_tensor.to('cuda'), k=10, algorithm="bruteforce-sharemem").cpu()
    # inject bidirectional edges from mesh and surface
    knn_graph.add_edges(srcs, dsts)
    # clean duplicattes and add self-loops
    graph = dgl.to_simple(knn_graph)
    graph = dgl.add_self_loop(graph)
    graph.edata.pop("count")

    # ------------------------------------------------------------------
    # Raw node fields
    # ------------------------------------------------------------------
    graph.ndata["distance_function"] = torch.tensor(dist_fn, dtype=torch.float32)
    graph.ndata["x-normals"] = torch.tensor(x_norm, dtype=torch.float32)
    graph.ndata["y-normals"] = torch.tensor(y_norm, dtype=torch.float32)

    # Dummy raw targets for compatibility
    zeros = torch.zeros(n, dtype=torch.float32)
    graph.ndata["x-velocity"] = zeros.clone()
    graph.ndata["y-velocity"] = zeros.clone()
    graph.ndata["pressure"] = zeros.clone()
    graph.ndata["turbulent_viscosity"] = zeros.clone()

    # ------------------------------------------------------------------
    # Reproduce node feature construction
    # ------------------------------------------------------------------
    x_vel_scalar = float(x_in[0])
    y_vel_scalar = float(y_in[0])
    vel_dir = np.full(n, np.arctan2(y_vel_scalar, x_vel_scalar), dtype=np.float32)

    vel_norm = np.sqrt(x_vel_scalar**2 + y_vel_scalar**2)
    if vel_norm < 1e-12:
        raise ValueError("Inlet velocity norm is zero; cannot define rotated coordinates")

    x_dir = np.array([x_vel_scalar / vel_norm, y_vel_scalar / vel_norm], dtype=np.float32)
    y_dir = np.array([-x_dir[1], x_dir[0]], dtype=np.float32)
    M = np.stack([x_dir, y_dir], axis=0)

    offset_x = x - 1.0
    offset_y = y
    coord = np.stack([offset_x, offset_y], axis=1).astype(np.float32)
    new_coord = (coord @ M.T) / 2.0

    coord_ = np.stack([x, y], axis=1).astype(np.float32)
    new_coord_ = (coord_ @ M.T) / 2.0

    surface_coord = coord[ordered_surface_ids]
    surface_normal = np.stack([x_norm[ordered_surface_ids], y_norm[ordered_surface_ids]], axis=1).astype(np.float32)

    if len(surface_coord) == 0:
        raise ValueError("No surface nodes found for nearest-surface feature construction")

    nbrs = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(surface_coord)
    distances_nn, indices = nbrs.kneighbors(coord)
    indices = indices.flatten().astype(np.int64) # (N,) array that maps each node to the closest surface node

    normal = surface_normal[indices].copy()

    # preserve the original code's special case
    special = indices == 0
    if np.any(special):
        normal[special] = coord[special]
        denom = np.linalg.norm(normal[special], axis=1, keepdims=True) + 1e-8
        normal[special] = -normal[special] / denom

    normal_dir = np.arctan2(normal[:, 1], normal[:, 0]).astype(np.float32)

    offset = coord - surface_coord[indices]
    dot_products = np.sum(offset * normal, axis=1)
    normal_len_sq = np.sum(normal**2, axis=1) + 1e-8
    projection_factors = dot_products / normal_len_sq

    offset = projection_factors[:, None] * normal
    rescaled_offset = (
        np.sign(projection_factors) * (np.abs(projection_factors) ** 0.2)
    )[:, None] * offset
    distances = np.linalg.norm(offset, axis=1).astype(np.float32)

    # base features (25-dim features per node)
    features = [
        x.astype(np.float32),                              # x-position
        y.astype(np.float32),                              # y-position
        ((x_in - NORMALIZE_PARAMS["means"][3]) / NORMALIZE_PARAMS["stds"][3]).astype(np.float32),
        ((y_in - NORMALIZE_PARAMS["means"][4]) / NORMALIZE_PARAMS["stds"][4]).astype(np.float32),
        dist_fn.astype(np.float32),                        # distance_function
        x_norm.astype(np.float32),                         # x-normals
        y_norm.astype(np.float32),                         # y-normals
        np.power(np.clip(dist_fn, 0.0, None), 0.2).astype(np.float32),
        # surface_from_normals.astype(np.float32),
        surface.astype(np.float32),                        # surface boolean
        new_coord[:, 0].astype(np.float32),
        new_coord[:, 1].astype(np.float32),
        new_coord_[:, 0].astype(np.float32),
        new_coord_[:, 1].astype(np.float32),
        normal[:, 0].astype(np.float32),
        normal[:, 1].astype(np.float32),
        offset[:, 0].astype(np.float32),
        offset[:, 1].astype(np.float32),
        rescaled_offset[:, 0].astype(np.float32),
        rescaled_offset[:, 1].astype(np.float32),
        distances.astype(np.float32),
        np.power(np.clip(distances, 0.0, None), 0.2).astype(np.float32),
        (indices / max(indices.max() + 1, 1)).astype(np.float32),
        vel_dir.astype(np.float32),
        normal_dir.astype(np.float32),
        (vel_dir - normal_dir).astype(np.float32),
    ]

    features = np.stack(features, axis=1)
    if features.shape[1] != 25:
        raise RuntimeError(f"Expected 25 node features, got {features.shape[1]}")

    graph.ndata["features"] = torch.tensor(features, dtype=torch.float32)
    graph.ndata["sim_ids"] = torch.zeros(graph.num_nodes(), dtype=torch.long)

    # Dummy normalized targets for compatibility
    targets = np.zeros((n, 4), dtype=np.float32)
    graph.ndata["targets"] = torch.tensor(targets, dtype=torch.float32)

    # Edge offsets
    src_t, dst_t = graph.edges()
    src_np = src_t.numpy()
    dst_np = dst_t.numpy()
    x_offset = x[dst_np] - x[src_np]
    y_offset = y[dst_np] - y[src_np]
    graph.edata["offsets"] = torch.tensor(
        np.stack([x_offset, y_offset], axis=1),
        dtype=torch.float32,
    )

    # Single-simulation shape feature
    shape_features = build_naca_shape_feature(AIRFOIL_PARAMS).unsqueeze(0)  # (1, 41)

    # Means/stds are not used by your inference path, but keep them in the tuple
    means = torch.zeros((1, 4), dtype=torch.float32)
    stds = torch.ones((1, 4), dtype=torch.float32)

    os.makedirs(os.path.dirname(output_pt_path), exist_ok=True)
    torch.save((graph, shape_features, means, stds), output_pt_path)

    print(f"Saved processed custom sample to: {output_pt_path}")
    print(graph)
    print("shape_features shape:", tuple(shape_features.shape))
    print("means shape:", tuple(means.shape))
    print("stds shape:", tuple(stds.shape))
    print("node features shape:", tuple(graph.ndata["features"].shape))
    print("edge offsets shape:", tuple(graph.edata["offsets"].shape))

In [3]:
preprocess_custom_sample(input_npy, processed_path)

Saved processed custom sample to: /orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/processed_data/processed_obgnn_closest_airfrans_testcase.pt
Graph(num_nodes=213508, num_edges=2688386,
      ndata_schemes={'distance_function': Scheme(shape=(), dtype=torch.float32), 'x-normals': Scheme(shape=(), dtype=torch.float32), 'y-normals': Scheme(shape=(), dtype=torch.float32), 'x-velocity': Scheme(shape=(), dtype=torch.float32), 'y-velocity': Scheme(shape=(), dtype=torch.float32), 'pressure': Scheme(shape=(), dtype=torch.float32), 'turbulent_viscosity': Scheme(shape=(), dtype=torch.float32), 'features': Scheme(shape=(25,), dtype=torch.float32), 'sim_ids': Scheme(shape=(), dtype=torch.int64), 'targets': Scheme(shape=(4,), dtype=torch.float32)}
      edata_schemes={'offsets': Scheme(shape=(2,), dtype=torch.float32)})
shape_features shape: (1, 41)
means shape: (1, 4)
stds shape: (1, 4)
node features shape: (213508, 25)
edge offsets shape: (2688386, 2)


## 3. Run Inference on Deviated Sample

In [ ]:
# Load preprocessed data
graph, shape_features, means, stds = torch.load(processed_path)

print(type(graph))
print(graph)
print(shape_features.shape, means.shape, stds.shape)

<class 'dgl.heterograph.DGLGraph'>
Graph(num_nodes=213508, num_edges=2688386,
      ndata_schemes={'distance_function': Scheme(shape=(), dtype=torch.float32), 'x-normals': Scheme(shape=(), dtype=torch.float32), 'y-normals': Scheme(shape=(), dtype=torch.float32), 'x-velocity': Scheme(shape=(), dtype=torch.float32), 'y-velocity': Scheme(shape=(), dtype=torch.float32), 'pressure': Scheme(shape=(), dtype=torch.float32), 'turbulent_viscosity': Scheme(shape=(), dtype=torch.float32), 'features': Scheme(shape=(25,), dtype=torch.float32), 'sim_ids': Scheme(shape=(), dtype=torch.int64), 'targets': Scheme(shape=(4,), dtype=torch.float32)}
      edata_schemes={'offsets': Scheme(shape=(2,), dtype=torch.float32)})
torch.Size([1, 41]) torch.Size([1, 4]) torch.Size([1, 4])


In [5]:
# Data Shape Checks
print("num_nodes:", graph.num_nodes())
print("num_edges:", graph.num_edges())

required_ndata = [
    "distance_function", "x-normals", "y-normals",
    "x-velocity", "y-velocity", "pressure", "turbulent_viscosity",
    "features", "sim_ids", "targets"
]
required_edata = ["offsets"]

print("Missing ndata:", [k for k in required_ndata if k not in graph.ndata])
print("Missing edata:", [k for k in required_edata if k not in graph.edata])

print("features:", graph.ndata["features"].shape)
print("sim_ids:", graph.ndata["sim_ids"].shape)
print("targets:", graph.ndata["targets"].shape)
print("offsets:", graph.edata["offsets"].shape)
print("shape_features:", shape_features.shape)

num_nodes: 213508
num_edges: 2688386
Missing ndata: []
Missing edata: []
features: torch.Size([213508, 25])
sim_ids: torch.Size([213508])
targets: torch.Size([213508, 4])
offsets: torch.Size([2688386, 2])
shape_features: torch.Size([1, 41])


In [ ]:
@torch.no_grad()
def run_inference_processed(model, graphs, shape_features, device, batch_size=8192, use_uva=True):
    """
    Runs inference on a processed merged DGL graph.

    Args:
        model: loaded OB-GNN model
        graphs: merged DGLGraph
        shape_features: torch.Tensor of shape (num_sims, 41)
        device: torch.device
        batch_size: dataloader batch size
        use_uva: whether to enable UVA in DGL dataloader

    Returns:
        pred_phys: torch.Tensor of shape (total_nodes, 4), in physical units
    """
    sampler = dgl.dataloading.MultiLayerNeighborSampler([15, 15])
    dataloader = dgl.dataloading.DataLoader(
        graphs,
        torch.arange(graphs.num_nodes()),
        sampler,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        use_uva=use_uva,
    )

    model.eval()
    model = model.to(device)
    shape_features = shape_features.to(device)

    outputs = []
    for it, (_, _, blocks) in enumerate(dataloader):
        blocks = [block.to(device) for block in blocks]

        x = blocks[0].srcdata["features"].to(device).to(torch.float32)
        sim_ids = blocks[0].srcdata["sim_ids"].to(device)
        x_shape = shape_features[sim_ids]
        x = torch.cat([x, x_shape], dim=1)

        y_hat = model(blocks, x)
        outputs.append(y_hat.cpu())

        if it % 100 == 0:
            print(f"Iteration {it:05d}/{len(dataloader):05d} | Inference")

    pred_norm = torch.cat(outputs, dim=0)  # (total_nodes, 4)

    pred_phys = pred_norm.clone()
    for i in range(4):
        std = float(NORMALIZE_PARAMS["stds"][i + 5])
        mean = float(NORMALIZE_PARAMS["means"][i + 5])
        pred_phys[:, i] = pred_phys[:, i] * std + mean

    return pred_phys


def save_single_prediction_tensor(pred_all, out_path):
    """
    Saves one prediction tensor of shape (N, 4) for a single custom simulation.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    if pred_all.ndim != 2 or pred_all.shape[1] != 4:
        raise ValueError(
            f"Expected prediction tensor of shape (N, 4), got {tuple(pred_all.shape)}"
        )
    tmp_path = out_path + ".tmp"
    torch.save(pred_all, tmp_path)
    os.replace(tmp_path, out_path)
    print(f"Saved prediction tensor -> {out_path}")


def run_inference_from_processed_file(
    model,
    processed_path,
    out_path,
    device,
    batch_size=8192,
    use_uva=True,
):
    """
    Loads a processed custom OB-GNN input file, runs inference, and saves predictions.

    Expected processed file format:
        (graphs, shape_features, means, stds)

    Args:
        model: loaded OB-GNN model
        processed_path: path to processed .pt file
        out_path: path to save predictions, e.g. pred_deviated.pt
        device: torch.device
        batch_size: inference batch size
        use_uva: whether to use UVA in DGL DataLoader

    Returns:
        pred_all: torch.Tensor of shape (N, 4) in physical units
    """
    if not os.path.exists(processed_path):
        raise FileNotFoundError(f"Processed file not found: {processed_path}")

    print(f"Loading processed file: {processed_path}")
    graphs, shape_features, means, stds = torch.load(processed_path)

    required_ndata = {"features", "sim_ids"}
    missing_ndata = required_ndata - set(graphs.ndata.keys())
    if missing_ndata:
        raise ValueError(f"Processed graph missing required ndata fields: {sorted(missing_ndata)}")
    if shape_features.ndim != 2 or shape_features.shape[1] != 41:
        raise ValueError(
            f"Expected shape_features to have shape (num_sims, 41), got {tuple(shape_features.shape)}"
        )
    if graphs.ndata["features"].shape[1] != 25:
        raise ValueError(
            f"Expected node feature dim 25, got {graphs.ndata['features'].shape[1]}"
        )

    unique_sim_ids = torch.unique(graphs.ndata["sim_ids"])
    print(f"Graph loaded: {graphs}")
    print(f"shape_features shape: {tuple(shape_features.shape)}")
    print(f"unique sim_ids: {unique_sim_ids.tolist()}")

    pred_all = run_inference_processed(
        model=model,
        graphs=graphs,
        shape_features=shape_features,
        device=device,
        batch_size=batch_size,
        use_uva=use_uva,
    )

    save_single_prediction_tensor(pred_all, out_path)
    return pred_all

In [ ]:
# Run inference and save prediction
device = torch.device("cuda:0")
model = load_model(CHECKPOINT_DIR, device)

pred_all = run_inference_from_processed_file(
    model=model,
    processed_path=processed_path,
    out_path=out_path,
    device=device,
    batch_size=8192,
    use_uva=True,
)

Namespace(epochs=12, steps=5000, hidden1=256, hidden2=256, hidden3=128, lr=0.00015, weight_decay=0.0, batch_size=128, w1=0.1, w2=0.2, w3=1.0, w4=0.00025, w5=0.1, w6=0.2, beta=1.0, noise1=1e-05, noise2=0.0, noise3=1e-05, bagging_k=1, k=1, lr_decay=0.99995, gpu=0, checkpoint_dir='./checkpoints')
Loading checkpoint: /orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/checkpoints/ckpt_bag00_ep0011.pt
Loading processed file: /orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/processed_data/processed_obgnn_closest_airfrans_testcase.pt
Graph loaded: Graph(num_nodes=213508, num_edges=2688386,
      ndata_schemes={'distance_function': Scheme(shape=(), dtype=torch.float32), 'x-normals': Scheme(shape=(), dtype=torch.float32), 'y-normals': Scheme(shape=(), dtype=torch.float32), 'x-velocity': Scheme(shape=(), dtype=torch.float32), 'y-velocity': Scheme(shape=(), dtype=torch.float32), 'pressure': Scheme(shape=(), dtype=torch.float32), 'turbulent_viscosity': Scheme(shape=(), dtype=torch

In [ ]:
# Check saved predictions
data = torch.load(out_path)
print(out_path)
print(data.shape)
print(data)

/orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/predictions/deviated/pred_obgnn_closest_airfrans_testcase.pt
torch.Size([213508, 4])
tensor([[ 5.4478e-02,  3.6339e-02,  8.8795e+02,  2.2760e-05],
        [ 9.0797e-02,  2.0385e-02,  2.2986e+02, -5.9097e-05],
        [ 5.4504e-02,  3.6345e-02,  8.8779e+02,  2.2764e-05],
        ...,
        [ 4.6233e+01,  2.2571e+00,  1.9466e+01, -2.4295e-04],
        [ 4.6312e+01,  2.2846e+00,  1.7767e+01, -2.4348e-04],
        [ 4.6409e+01,  2.3199e+00,  1.6129e+01, -2.4348e-04]])


## 4. Save Prediction Plots

In [ ]:
def save_prediction_plots_from_xy_edges(
    pred_pt_path,
    x,
    y,
    edges,
    out_dir,
    sample_name="deviated",
    crop_xlim=None,
    crop_ylim=None,
    show_edges=True,
    edge_linewidth=0.15,
    edge_alpha=0.25,
    node_size=3,
    dpi=200,
):
    """
    Visualize a single custom sample using node-colored scatter and optional mesh edges.

    Args:
        pred_pt_path: path to saved prediction tensor of shape (N, 4)
        x, y: numpy arrays of shape (N,)
        edges: numpy array of shape (E, 2), undirected mesh edges
        out_dir: folder to save images
        sample_name: prefix for saved file names
        crop_xlim, crop_ylim: optional axis limits
        show_edges: whether to draw mesh edges
        edge_linewidth: linewidth for edge overlay
        edge_alpha: transparency for edge overlay
        node_size: marker size for scatter
        dpi: saved figure resolution
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pred_tensor = torch.load(pred_pt_path, map_location="cpu")
    # pred_tensor = np.load(pred_pt_path)

    pred = pred_tensor.numpy() if isinstance(pred_tensor, torch.Tensor) else np.asarray(pred_tensor)

    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    edges = np.asarray(edges, dtype=np.int64)

    if pred.ndim != 2 or pred.shape[1] != 4:
        raise ValueError(f"Expected prediction tensor shape (N, 4), got {pred.shape}")
    if x.shape != y.shape:
        raise ValueError(f"x and y must have same shape, got {x.shape} and {y.shape}")
    if pred.shape[0] != x.shape[0]:
        raise ValueError(f"Prediction node count {pred.shape[0]} does not match x/y node count {x.shape[0]}")
    if edges.ndim != 2 or edges.shape[1] != 2:
        raise ValueError(f"edges must have shape (E, 2), got {edges.shape}")

    field_map = {
        0: "x_velocity",
        1: "y_velocity",
        2: "pressure",
        3: "turbulent_viscosity",
    }

    # Build line segments once
    segments = None
    if show_edges:
        segments = np.stack([
            np.column_stack([x[edges[:, 0]], y[edges[:, 0]]]),
            np.column_stack([x[edges[:, 1]], y[edges[:, 1]]]),
        ], axis=1)  # (E, 2, 2)

    for field_col, field_name in field_map.items():
        values = pred[:, field_col]

        fig, ax = plt.subplots(figsize=(8, 4.5))

        if show_edges:
            lc = LineCollection(
                segments,
                linewidths=edge_linewidth,
                alpha=edge_alpha,
            )
            ax.add_collection(lc)

        sc = ax.scatter(x, y, c=values, s=node_size)
        plt.colorbar(sc, ax=ax)

        ax.set_title(f"Prediction: {field_name}")
        ax.set_aspect("equal")

        if crop_xlim is not None:
            ax.set_xlim(*crop_xlim)
        if crop_ylim is not None:
            ax.set_ylim(*crop_ylim)

        plt.tight_layout()
        save_path = out_dir / f"{sample_name}_{field_name}.png"
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)

        print(f"saved {save_path}")

In [ ]:
# get mesh positions & edges from input
d = np.load(input_npy, allow_pickle=True)
x=d[0]
y=d[1]
edges = d[-1]

# save plots
pred_pt_path = out_path
save_prediction_plots_from_xy_edges(
    pred_pt_path=pred_pt_path,
    x=x,
    y=y,
    edges=edges,
    out_dir=out_dir,
    sample_name=SAMPLE_NAME,
    crop_xlim=(-1, 3),
    crop_ylim=(-1, 1),
    show_edges=True,
    edge_linewidth=0.1,
    edge_alpha=0.2,
    node_size=2,
    dpi=200,
)

saved /orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/prediction_plots/nominal/obgnn_closest_airfrans_testcase_x_velocity.png
saved /orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/prediction_plots/nominal/obgnn_closest_airfrans_testcase_y_velocity.png
saved /orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/prediction_plots/nominal/obgnn_closest_airfrans_testcase_pressure.png
saved /orcd/home/002/jlu25/ML4CFD-Offset-based-Graph-Convolution/prediction_plots/nominal/obgnn_closest_airfrans_testcase_turbulent_viscosity.png
